In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


**Prepare Data**

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV


train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")



**Feature Engineering**

In [3]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)

rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major',
               'Rev','Sir','Jonkheer','Dona']

train["Title"] = train["Title"].replace(rare_titles, "Rare")
test["Title"] = test["Title"].replace(rare_titles, "Rare")

train["Title"] = train["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})
test["Title"] = test["Title"].replace({"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"})

train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

train["AgeGroup"] = pd.cut(train["Age"], bins=[0,12,18,35,60,100])
test["AgeGroup"] = pd.cut(test["Age"], bins=[0,12,18,35,60,100])

train["Fare"] = np.log1p(train["Fare"])
test["Fare"] = np.log1p(test["Fare"])

features = ["Pclass","Sex","Age","Fare","Embarked",
            "FamilySize","Title","IsAlone","SibSp","Parch","AgeGroup"]

y = train["Survived"]
X = train[features]

numeric_features = ["Age", "Fare","FamilySize","SibSp","Parch"]
categorical_features = ["Pclass", "Sex", "Embarked","Title","IsAlone","AgeGroup"]

numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)



In [4]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

grid = GridSearchCV(
    model,
    param_grid = {
        "classifier__n_estimators": [300, 500, 800],
        "classifier__max_depth": [4, 5, 6, None],
        "classifier__min_samples_leaf": [1, 2, 3]
    },
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid.fit(X, y)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

best_model = grid.best_estimator_

best_model.fit(X, y)

predict_ans = best_model.predict(test[features])

output = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predict_ans
})

output.to_csv("submission.csv", index=False)


Best parameters: {'classifier__max_depth': 4, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 300}
Best CV score: 0.832741196409516
